# How Fast Do Markets React to Trade Announcements?

In this notebook you'll look through the US-China trade war in the lens of
the financial market.

Almost everything is pre-written, and in
marked spots you'll edit a date, a number, or a list of stock tickers.

---

### Step 1

Watch the intro video:

This notebook is your personal copy and your edits save
automatically. If it ever gets hopelessly tangled, **Ctrl+Z / Cmd+Z** inside a cell
fixes almost everything but feel free to message/email Heagen for a new one or any questions if needed.

### How it works

- Cells run top to bottom: click a cell, press **Shift+Enter** (or ▶).
- **JUST RUN** cells: run, don't edit.
- **EDIT** cells: change the line marked `# EDIT`.

## Setup

**JUST RUN**: loads tools and three and a half years of daily stock prices
(2017 through mid-2020)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_URL = "https://raw.githubusercontent.com/heagenb03/twl-hs-interns-notebooks/main/data/tariff_prices.csv"
prices = pd.read_csv(DATA_URL, index_col="date", parse_dates=True)

print(f"Loaded {len(prices)} trading days for {len(prices.columns)} tickers: {list(prices.columns)}")
prices.head()

 > **What you should see:** `Loaded 879 trading days for 6 tickers: ['SPY', 'FXI',
> 'BA', 'CAT', 'SOYB', 'AAPL']` and a table of prices with dates down the side.

### What each one is

| Ticker | What it is | Why it's here |
|---|---|---|
| **SPY** | The S&P 500 (i.e. top 500 U.S. companies) | Our baseline |
| **FXI** | China Large-cap ETF (i.e. top 50 China companies) | Direct China exposure |
| **BA** | Boeing | Biggest export customer for China|
| **CAT** | Caterpillar | Global industrial bellwether (i.e. a company that can help determine the "health" of a market)|
| **SOYB** | Soybeans | China's retaliation target to hit U.S. farmers|
| **AAPL** | Apple | Builds and sells in China |

The six announcement dates we've pre-loaded. **JUST RUN:**

In [ ]:
EVENTS = {
    "2018-03-22": "Tariff memo signed (escalation, surprise)",
    "2018-06-15": "25% tariffs on $50B announced (escalation, expected)",
    "2018-12-03": "Buenos Aires truce (de-escalation)",
    "2019-05-06": "Sunday tweet: tariffs to 25% (escalation, surprise)",
    "2019-08-23": "China retaliates; companies 'ordered' out (escalation, surprise)",
    "2020-01-15": "Phase One deal signed (de-escalation, fully expected)",
}
for d, label in EVENTS.items():
    print(f"{d}: {label}")

Setup cell for plotting. **JUST RUN:**

In [ ]:
def _event_day(date):
    date = pd.Timestamp(str(date))
    if date not in prices.index:
        later = prices.index[prices.index >= date]
        if len(later) == 0:
            print(f"({date.date()} is past the end of this dataset.)"); return None
        print(f"({date.date()} wasn't a trading day, using {later[0].date()} instead.)")
        return later[0]
    return date

def event_plot(date, window, tickers=["SPY"]):
    """Prices around an announcement, rebased to 100 the day before it."""
    if isinstance(tickers, str):
        tickers = [tickers]
    window = int(window)
    day0 = _event_day(date)
    if day0 is None: return
    i = prices.index.get_loc(day0)
    if i == 0:
        print(f"({date} is the first day in this dataset, so there's no day before it "
              f"to compare against. Try a later date.)")
        return
    chunk = prices.iloc[max(i - window, 0): i + window + 1]
    fig, ax = plt.subplots(figsize=(10, 5))
    for tk in tickers:
        if tk not in prices.columns:
            print(f"('{tk}' isn't in this dataset — options: {list(prices.columns)})"); continue
        rebased = chunk[tk] / prices[tk].iloc[i - 1] * 100
        ax.plot(range(-min(window, i), len(chunk) - min(window, i)), rebased, marker="o",
                markersize=3, label=tk)
        day0_ret = (prices[tk].iloc[i] / prices[tk].iloc[i - 1] - 1) * 100
        print(f"{tk}: day-0 move {day0_ret:+.1f}%")
    ax.axvline(0, color="red", linestyle="--", alpha=0.6, label="announcement day")
    label = EVENTS.get(str(date), str(date))
    ax.set_title(label); ax.set_xlabel("trading days from announcement")
    ax.set_ylabel("price (100 = day before)")
    ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

def compare_events(date_a, date_b, ticker="SPY", window=5):
    """Overlay one ticker's path around two different announcements."""
    if ticker not in prices.columns:
        print(f"('{ticker}' isn't in this dataset. Options: {list(prices.columns)})"); return
    window = int(window)
    fig, ax = plt.subplots(figsize=(10, 5))
    for date in [date_a, date_b]:
        day0 = _event_day(date)
        if day0 is None: return
        i = prices.index.get_loc(day0)
        if i == 0:
            print(f"({date} is the first day in this dataset, so there's no day before it "
                  f"to compare against. Try a later date.)")
            return
        chunk = prices[ticker].iloc[max(i - window, 0): i + window + 1]
        rebased = chunk / prices[ticker].iloc[i - 1] * 100
        ax.plot(range(-min(window, i), len(chunk) - min(window, i)), rebased, marker="o", markersize=4,
                label=f"{date}: {EVENTS.get(str(date), '')[:38]}")
        day0_ret = (prices[ticker].iloc[i] / prices[ticker].iloc[i - 1] - 1) * 100
        print(f"{date} — {ticker} day-0 move: {day0_ret:+.1f}%")
    ax.axvline(0, color="red", linestyle="--", alpha=0.6)
    ax.set_title(f"{ticker} around two announcements")
    ax.set_xlabel("trading days from announcement"); ax.set_ylabel("price (100 = day before)")
    ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

def scoreboard(tickers=["SPY", "FXI", "BA", "SOYB"]):
    """Day-0 percent move for every pre-loaded event, per ticker."""
    rows = []
    for d, label in EVENTS.items():
        day0 = _event_day(d)
        if day0 is None: continue
        i = prices.index.get_loc(day0)
        row = {"date": d, "event": label[:45]}
        for tk in tickers:
            row[tk] = f"{(prices[tk].iloc[i] / prices[tk].iloc[i-1] - 1) * 100:+.1f}%"
        rows.append(row)
    return pd.DataFrame(rows)

def market_adjusted(ticker, date, window=5):
    """Stretch helper: a stock's move MINUS the market's move (its 'abnormal' return)."""
    if ticker not in prices.columns:
        print(f"('{ticker}' isn't in this dataset. Options: {list(prices.columns)})"); return
    window = int(window)
    day0 = _event_day(date)
    if day0 is None: return
    i = prices.index.get_loc(day0)
    if i == 0:
        print(f"({date} is the first day in this dataset, so there's no day before it "
              f"to compare against. Try a later date.)")
        return
    stock = prices[ticker].iloc[i - 1: i + window + 1].pct_change().dropna()
    mkt = prices["SPY"].iloc[i - 1: i + window + 1].pct_change().dropna()
    abnormal = ((stock - mkt) * 100).cumsum()
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(range(len(abnormal)), abnormal.values, marker="o")
    ax.axhline(0, color="gray", linestyle="--")
    ax.set_title(f"{ticker}'s move BEYOND the market, after {date}")
    ax.set_xlabel("trading days from announcement"); ax.set_ylabel("cumulative abnormal return (%)")
    ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
    print(f"After {window} days, {ticker} moved {abnormal.iloc[-1]:+.1f}% beyond what the market explains.")

print("Done")

---
## How Fast?

August 23, 2019: China announces retaliation tariffs, and the president tweets that
American companies are "hereby ordered" to leave China.

**EDIT:** run as-is first, then change the date (copy any from the EVENTS
list) and the window.

In [ ]:
announcement_date = "2018-03-22"   # <-- EDIT (copy a date from the EVENTS list above)
window_in_days = 100            # <-- EDIT (trading days shown on each side)

event_plot(announcement_date, window_in_days)

> **What you should see** (with the starting values): a price line that's along 100, then **goes down right at trading day 0**, with a printed
> `SPY: day-0 move -2.6%`.

### Things to try

1. **Widen the window to 40.** Does the drop get spread out? Or in other words did the market look like it expected it?
2. Try `"2018-03-22"`, the very first big tariff announcement. Compare the size of its cliff to August 2019's.
3. Look at the days before day 0. Any hint of the drop coming? Does it look like the markets moved before the announcements are made?

---
## Ups and Downs

The trade war had escalations and de-escalations, so does the market generally treat good and bad news equally?

**EDIT:** compare an escalation to a de-escalation.

In [ ]:
first_event  = "2019-08-23"   # <-- EDIT (an escalation)
second_event = "2018-12-03"   # <-- EDIT (the Buenos Aires truce)

compare_events(first_event, second_event)

> **What you should see** (with the starting dates): two overlaid lines of SPY's price and their percentage change (`-2.6%` versus `+1.3%`).

### Things to try

1. Swap in with the other escalations. Is the pattern general? Do drops on bad news run bigger than pops on good news?

2. Why might that be? Do you think this generalizes the behavior of the S&P as a whole? Even the broader market from bonds, currencies, or commodities?

3. Run the same comparison with `ticker="SOYB"` added inside the parentheses:
   `compare_events(first_event, second_event, ticker="SOYB")`. How different does it look?

---
## Surpise or unexpected?

May 5, 2019: the **Sunday
tweet** that re-escalatess the trade war. January 15, 2020: the **Phase One
deal** that was negotiated publicly for months with a signing date known well for the public.

**EDIT:**

In [ ]:
surprise_event = "2019-05-06"   # <-- EDIT
expected_event = "2020-01-15"   # <-- EDIT
tickers = ["SPY", "FXI", "BA"]   # <-- EDIT this list

event_plot(surprise_event, 5, tickers)
event_plot(expected_event, 5, tickers)

> **What you should see** (with the starting values): the first chart drops at trading day-0 with a `-0.4%` to `-2.6%` move depending on ticker. The second
> chart, the biggest deal of the trade war, has little move: day-0 moves
> around `-0.8%` to `+0.2%`.

### Things to try

1. Signing the deal that paused the trade war moved markets
   less than a single tweet compared to a major resolution on January 15 when everyone already knew the
   deal was coming. So everyone who wanted to buy on the news already bought
   weeks earlier because the information entered prices when it became known not when it became offical. (For technical speak this is an asset "pricing in" the future price of itself based on the information it currently has)

2. Swap the tickers list to `["SOYB", "CAT", "AAPL"]` and re-run both. Which companies had the most riding on surprise?

3. Re-read the EVENTS list then rank the
   six by how surprising they were. Finally check your ranking with the scoreboard below.

**JUST RUN:**

In [ ]:
# The day-0 moves for all six events
scoreboard()

> **What you should see:** a 6-row table, one event per row, with signed percentage moves per ticker.

---
## Market Adjusted

When Boeing drops 4% on an announcement day, how much of that is Boeing itself v.s. how much is just everything else dropping? We can answer this by subtracting the market's move, so what's left (the difference) is the stocks abnormal return.

**EDIT:**

In [ ]:
ticker = "BA"  # <-- EDIT
event  = "2019-08-23"  # <-- EDIT

market_adjusted(ticker, event)

> **What you should see:** The market adjusted return and a
> printed line describing it: `After 5 days, BA moved +2.6% beyond what the market explains.`

Try every ticker on the same event. Which company's fate was most tied to that
specific announcement, once you strip out the marketwide panic? Does the answer
match what you'd guess?

---
## Recap

With edited strings and lists in Python in a Jupyter Notebook:

- watched news hit prices across different assets
- compare how markets punish escalation v.s. rewarding peace
- the definition of pricing something in
- computed abnormal returns

With what you know about current/hear about markets generally, do you see this behavior in the markets consistently?

Do you think you could you use data and code like this to analyze or find the future price of an asset?

With the rapid expansion of technology, do you think technology is making the market better? Is a world where all trades are done by different models/computers trained on this type of data a better one?

Lastly any thoughts, ideas, concerns on this assignment as a whole?